In [1]:
import os
import re
import time
import ee
import numpy as np
import pandas as pd

PROJECT_ID = "acoustic-gizmo-513018-a8"
ee.Initialize(project=PROJECT_ID)

HUBS = [
    ("Kafr_ElSheikh", 31.11, 30.94), ("Dakahlia", 31.04, 31.38),
    ("Sharqia", 30.72, 31.67), ("Beheira", 31.03, 30.47),
    ("Gharbia", 30.79, 31.00), ("Menoufia", 30.55, 31.01),
    ("Qalyubia", 30.46, 31.18), ("Salhia_Reclaimed", 30.65, 31.95),
    ("Fayoum", 29.31, 30.84), ("Beni_Suef", 29.07, 31.09),
    ("Minya", 28.09, 30.75), ("Assiut", 27.18, 31.18),
    ("Sohag", 26.56, 31.69), ("Qena_Luxor", 25.69, 32.64),
]

hubs_fc = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([lon, lat]), {"Governorate": name})
    for name, lat, lon in HUBS
])


def safe_mean(col, band_names):
    dummy = ee.Image.constant([0] * len(band_names)).toFloat().rename(band_names).updateMask(ee.Image.constant(0))
    return col.map(lambda img: img.toFloat()).merge(ee.ImageCollection([dummy])).mean()


# 1. الخصائص الثابتة للتضاريس والكتلة الحيوية والتربة
srtm = ee.Image("USGS/SRTMGL1_003")
static_img = (
    srtm.select("elevation").rename("SRTM_Elevation_m")
    .addBands(ee.Terrain.slope(srtm).rename("SRTM_Slope_Deg"))
    .addBands(ee.Image("LARSE/GEDI/GEDI04_B_002").select("MU").rename("GEDI_Biomass_Mg_ha"))
    .addBands(ee.Image("OpenLandMap/SOL/SOL_PH-H2O_USDA-4C1A2A_M/v02").select("b0").multiply(0.1).rename("Soil_pH_Topsoil"))
    .addBands(ee.Image("OpenLandMap/SOL/SOL_ORGANIC-CARBON_USDA-6A1C_M/v02").select("b0").multiply(5.0).rename("Soil_Organic_Carbon_g_kg"))
    .addBands(ee.Image("OpenLandMap/SOL/SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02").select("b0").rename("Soil_Clay_Pct"))
    .addBands(ee.Image("OpenLandMap/SOL/SOL_SAND-WFRACTION_USDA-3A1A1A_M/v02").select("b0").rename("Soil_Sand_Pct"))
)

static_res = static_img.reduceRegions(collection=hubs_fc, reducer=ee.Reducer.first(), scale=250).getInfo()
df_static = pd.DataFrame([f["properties"] for f in static_res["features"]])

# 2. السلاسل الزمنية لنموذج سطح الأرض NASA GLDAS
BANDS = ["GLDAS_SoilMoi_0_10cm", "GLDAS_SoilMoi_10_40cm", "GLDAS_SoilMoi_40_100cm", "GLDAS_Evap_Rate", "GLDAS_Canopy_Water"]
gldas_col = ee.ImageCollection("NASA/GLDAS/V021/NOAH/G025/T3H").select(
    ["SoilMoi0_10cm_inst", "SoilMoi10_40cm_inst", "SoilMoi40_100cm_inst", "Evap_tavg", "CanopInt_inst"], BANDS
)

date_bins = pd.date_range("2016-01-01", "2025-12-31", freq="10D")
expected_dates = pd.to_datetime(date_bins[:-1])
intervals = [(date_bins[i].strftime("%Y-%m-%d"), date_bins[i + 1].strftime("%Y-%m-%d")) for i in range(len(date_bins) - 1)]

CHUNKS = [
    ("2016-01-01", "2017-12-31"),
    ("2018-01-01", "2019-12-31"),
    ("2020-01-01", "2021-12-31"),
    ("2022-01-01", "2023-12-31"),
    ("2024-01-01", "2025-12-31"),
]

records_dict = {}
print("📥 [6/6] بدء سحب بيانات NASA GLDAS وخصائص التربة (وضع توفير الكوتا)...", flush=True)

for s_c, e_c in CHUNKS:
    sub_int = [x for x in intervals if s_c <= x[0] <= e_c]
    stack = ee.ImageCollection([
        safe_mean(gldas_col.filterDate(s, e), BANDS).rename([f"{s}_{b}" for b in BANDS])
        for s, e in sub_int
    ]).toBands()

    res = stack.reduceRegions(collection=hubs_fc, reducer=ee.Reducer.first(), scale=25000).getInfo()
    for feat in res["features"]:
        props = feat["properties"]
        gov = props["Governorate"]
        for k, val in props.items():
            m = re.match(r"^(?:\d+_)?(\d{4}-\d{2}-\d{2})_(.+)$", k)
            if m:
                records_dict.setdefault((gov, m.group(1)), {})[m.group(2)] = val
    print(f"   ✓ GLDAS: اكتملت الفترة {s_c[:4]}–{e_c[:4]}", flush=True)
    time.sleep(1.5)

df_raw = pd.DataFrame([{"Governorate": gov, "Date": dt, **vals} for (gov, dt), vals in records_dict.items()])
df_raw["Date"] = pd.to_datetime(df_raw["Date"])

grid = pd.MultiIndex.from_product([[h[0] for h in HUBS], expected_dates], names=["Governorate", "Date"]).to_frame().reset_index(drop=True)
out = pd.merge(grid, df_raw, on=["Governorate", "Date"], how="left")
out = pd.merge(out, df_static, on="Governorate", how="left").sort_values(["Governorate", "Date"]).reset_index(drop=True)

num_cols = out.select_dtypes(include=[np.number]).columns
out[num_cols] = out.groupby("Governorate")[num_cols].transform(lambda g: g.interpolate("linear").ffill().bfill())
out["Date"] = out["Date"].dt.strftime("%Y-%m-%d")

output_path = r"d:\vscode\NASA\DATA\06_GLDAS_Terrain_Soil.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
out.to_csv(output_path, index=False)
print(f"✅ اكتمل الملف السادس: {output_path} | الأبعاد: {out.shape}\n")

C:\Users\eng mohamed\AppData\Roaming\Python\Python314\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


📥 [6/6] بدء سحب بيانات NASA GLDAS وخصائص التربة (وضع توفير الكوتا)...


C:\Users\eng mohamed\AppData\Roaming\Python\Python314\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


   ✓ GLDAS: اكتملت الفترة 2016–2017


C:\Users\eng mohamed\AppData\Roaming\Python\Python314\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


   ✓ GLDAS: اكتملت الفترة 2018–2019
   ✓ GLDAS: اكتملت الفترة 2020–2021
   ✓ GLDAS: اكتملت الفترة 2022–2023
   ✓ GLDAS: اكتملت الفترة 2024–2025
✅ اكتمل الملف السادس: d:\vscode\NASA\DATA\06_GLDAS_Terrain_Soil.csv | الأبعاد: (5110, 14)

